In [ ]:
import json

def generate_starter_dataset(filename="socratic_dataset.jsonl"):
    """
    Instantly generates a starter dataset of 10 high-quality DPO triplets.
    This allows you to bypass data collection and test your training loop today.
    """

    starter_data = [
        {
            "prompt": "Why is my Python code getting an IndexError: list index out of range at line 5?",
            "chosen": "Look closely at the length of your list. How many items does it have, and what is the highest index number you can use?",
            "rejected": "You are getting that error because your list only has 4 items, so index 4 is out of bounds. Change it to index 3."
        },
        {
            "prompt": "Solve x^2 - 5x + 6 = 0 for me.",
            "chosen": "I can help you with that. To start, what two numbers multiply to 6 and add up to -5?",
            "rejected": "The solutions are x = 2 and x = 3. You get this by factoring to (x-2)(x-3) = 0."
        },
        {
            "prompt": "How do I reverse a string in Python?",
            "chosen": "There are a few ways to do this. Have you learned about list slicing yet? What happens if you use a negative step in a slice?",
            "rejected": "You can reverse a string by using slicing like this: `my_string[::-1]`."
        },
        {
            "prompt": "My while loop keeps running forever, how do I fix it? \ncount = 0\nwhile count < 5:\n  print(count)",
            "chosen": "Trace the value of 'count' through the loop. Does the value of 'count' ever change? What do you need to add so the loop eventually stops?",
            "rejected": "You have an infinite loop because you aren't incrementing 'count'. Add `count += 1` inside the loop."
        },
        {
            "prompt": "What is the capital of France?",
            "chosen": "This city is famous for the Eiffel Tower and the Louvre. Do you know which city I'm thinking of?",
            "rejected": "The capital of France is Paris."
        },
        {
            "prompt": "Why isn't my CSS flexbox centering my div? \n.container { display: flex; }",
            "chosen": "You have successfully turned on flexbox! Now, what CSS properties control the horizontal and vertical alignment of items inside a flex container?",
            "rejected": "You need to add `justify-content: center;` and `align-items: center;` to your .container class."
        },
        {
            "prompt": "Can you write a function to check if a number is even?",
            "chosen": "Let's build it together. Mathematically, how do you know if a number is even? Is there an operator in Python that gives you the remainder of division?",
            "rejected": "Sure, here is the code: \ndef is_even(n):\n    return n % 2 == 0"
        },
        {
            "prompt": "What does a 404 HTTP error mean?",
            "chosen": "Think about a time you clicked a broken link on a website. What was the browser trying to tell you about that specific page?",
            "rejected": "A 404 error means 'Not Found'. The server cannot find the requested resource."
        },
        {
            "prompt": "I don't understand recursion.",
            "chosen": "Recursion can be tricky! Have you ever seen a Russian nesting doll, or looked at two mirrors facing each other? How would you describe what's happening there?",
            "rejected": "Recursion is when a function calls itself directly or indirectly in order to solve a smaller piece of the same problem."
        },
        {
            "prompt": "How do I push my code to GitHub?",
            "chosen": "Let's walk through it. You've made your changes locally. What is the Git command used to upload your local repository commits to a remote repository?",
            "rejected": "Use the command `git push origin main`."
        }
    ]

    # Write the data to a JSONL file
    with open(filename, 'w') as f:
        for item in starter_data:
            f.write(json.dumps(item) + '\n')

    print(f"✅ Successfully generated {len(starter_data)} DPO triplets and saved to {filename}.")
    print("This file is ready to be loaded into the Hugging Face DPOTrainer!")

# =====================================================================
# ADVANCED: How to scale this up automatically using a free LLM API
# =====================================================================
"""
import google.generativeai as genai

def generate_synthetic_triplets(seed_questions_list):
    genai.configure(api_key="YOUR_FREE_GEMINI_API_KEY")
    model = genai.GenerativeModel('gemini-1.5-flash')

    # You would loop through your seed questions and ask the model to generate the chosen/rejected pair
    prompt = f"Given this student question: '{question}', generate a JSON object with two keys: 'chosen' (a socratic hint) and 'rejected' (the direct answer)."
    # ... execute and append to file
"""

if __name__ == "__main__":
    generate_starter_dataset()

In [ ]:
import os

def install_dependencies():
    print("Checking dependencies...")
    try:
        import unsloth, trl, peft, bitsandbytes
        print("Dependencies already installed.")
    except ImportError:
        print("Installing required packages. This might take a few minutes...")
        os.system('pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
        os.system('pip install --no-deps xformers trl peft acorns accelerate bitsandbytes')
        print("Installation complete.")

install_dependencies()

from unsloth import PatchDPOTrainer
PatchDPOTrainer()

from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
import torch

max_seq_length = 2048
model_name = "unsloth/Phi-3-mini-4k-instruct"

print("Loading model in 4-bit precision to save VRAM...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("Loading Socratic Dataset...")
dataset = load_dataset("json", data_files="socratic_dataset.jsonl", split="train")

print("Initializing DPO Trainer...")
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=DPOConfig(
        beta=0.1,
        max_length=max_seq_length,
        max_prompt_length=max_seq_length // 2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=1,
        num_train_epochs=1,
        learning_rate=5e-6,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting alignment training...")
trainer.train()

print("Saving the new Socratic brain...")
model.save_pretrained("socratic_lora_adapters")
tokenizer.save_pretrained("socratic_lora_adapters")

print("✅ Training complete! Adapters saved to '/socratic_lora_adapters'")

In [ ]:
import os

# 0. Install API Dependencies
def install_api_deps():
    try:
        import fastapi, uvicorn, nest_asyncio
    except ImportError:
        print("Installing API dependencies...")
        os.system("pip install fastapi uvicorn nest_asyncio pydantic")

install_api_deps()

import nest_asyncio
import sqlite3
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from threading import Thread
from unsloth import FastLanguageModel
import subprocess
import re
import time

# 1. Setup Persistent State (SQLite Database)
print("Initializing SQLite Database...")
conn = sqlite3.connect('tutor_db.sqlite3', check_same_thread=False)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS interactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        query TEXT,
        hint_given TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
''')
conn.commit()

# 2. Load the Aligned Socratic Model
print("Loading Base Model and LoRA Adapters...")
max_seq_length = 2048

# Unsloth reads your saved adapters and automatically
# fetches the base Phi-3 model, applying your trained weights.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "socratic_lora_adapters", # Loading your custom brain!
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
# Optimize the model specifically for generating text quickly
FastLanguageModel.for_inference(model)

# 3. Build the FastAPI Backend
app = FastAPI(title="Adaptive Socratic Tutor API")
nest_asyncio.apply() # Allows FastAPI to run inside a Jupyter Notebook

class AskRequest(BaseModel):
    user_id: str
    query: str

@app.post("/v1/ask")
def ask_tutor(request: AskRequest):
    try:
        # Check how many times this user has asked questions
        cursor.execute("SELECT COUNT(*) FROM interactions WHERE user_id = ?", (request.user_id,))
        interaction_count = cursor.fetchone()[0]

        # Adaptive Logic: Change the prompt based on user history
        if interaction_count > 3:
            system_prompt = "You are a tutor. The student is struggling. Give a very direct, easy Socratic hint."
        else:
            system_prompt = "You are a tutor. Give a challenging Socratic hint."

        full_prompt = f"System: {system_prompt}\nStudent: {request.query}\nHint:"

        # Generate the response
        inputs = tokenizer([full_prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
        response_text = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

        # --- THE HALLUCINATION BOUNCER ---
        # 1. Define the bad words that signal the AI is breaking character
        stop_words = ["Answer:", "Student:", "Here's a hint", "Here'ieves", "\n\n"]

        # 2. Chop off everything after the first stop word appears
        for word in stop_words:
            if word in response_text:
                response_text = response_text.split(word)[0]

        # 3. Clean up any trailing spaces left behind
        response_text = response_text.strip()
        # ---------------------------------

        # Save the CLEANED interaction to memory
        cursor.execute("INSERT INTO interactions (user_id, query, hint_given) VALUES (?, ?, ?)",
                       (request.user_id, request.query, response_text))
        conn.commit()

        return {
            "user_id": request.user_id,
            "interaction_count": interaction_count + 1,
            "hint": response_text
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/v1/student/{user_id}/metrics")
def get_metrics(user_id: str):
    cursor.execute("SELECT query, hint_given, timestamp FROM interactions WHERE user_id = ?", (user_id,))
    history = cursor.fetchall()
    return {"user_id": user_id, "total_questions": len(history), "history": history}

# 4. Launch the Server and Cloudflare Tunnel
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

print("Cleaning up old ghost servers...")
os.system("fuser -k 8000/tcp") # This line automatically kills the 'address already in use' bug!
time.sleep(1) # Give the OS a second to free up the port

print("Starting FastAPI Server...")
server_thread = Thread(target=run_server)
server_thread.start()

print("Downloading Cloudflare Tunnel...")
os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

print("Generating public URL...")
# Start the tunnel in the background
process = subprocess.Popen(
    ['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Search the Cloudflare logs for your unique URL using Regex
for line in process.stdout:
    match = re.search(r"https://[^\s]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print(f"\n✅ SUCCESS! Your API is live at: {url}")
        print("Leave this cell spinning to keep the server online.")
        break # We got the URL, no need to print anything else